In [22]:
import os
import numpy as np
from PIL import Image

# Set paths
base_dir = "E:/cervical image"
folders = ['normal', 'abnormal']
target_size = (132, 132)  # (width, height)

# Initialize storage
all_images = []
all_labels = []

for label, folder in enumerate(folders):
    path = os.path.join(base_dir, folder)
    for filename in os.listdir(path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(path, filename)

            # Load, convert, resize, normalize
            img = Image.open(img_path).convert("RGB")
            img = img.resize(target_size)
            all_images.append(np.array(img, dtype=np.float32) / 255.0)
            all_labels.append(label)

# Convert to NumPy arrays
image_array = np.array(all_images)
label_array = np.array(all_labels)

# Save arrays
np.save('resized_images.npy', image_array)
np.save('labels.npy', label_array)

print(f"✅ Done! Processed {len(all_images)} images.")


KeyboardInterrupt



In [3]:
import numpy as np
from sklearn.model_selection import train_test_split 

images = np.load('resized_images.npy')
labels = np.load('labels.npy')

# Split 80% train, 20% test
x_train, x_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels)

In [4]:
print(images.shape)

(332, 132, 132, 3)


In [5]:
x_train = x_train / 255.0
x_test = x_test / 255.0

In [6]:
from tensorflow.keras import models
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

In [7]:
def build_lenet():
    model = models.Sequential([
        layers.Conv2D(6, (5, 5), activation='relu', input_shape=(132, 132, 3)),
        layers.AveragePooling2D(pool_size=(2, 2)),
        layers.Conv2D(16, (5, 5), activation='relu'),
        layers.AveragePooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(120, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model
    

In [8]:
from tensorflow.keras import layers, models

input_tensor = layers.Input(shape=(132, 132, 3))  # or whatever your actual input shape is
x = layers.Conv2D(32, (3, 3), activation='relu')(input_tensor)

In [9]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout

x = Conv2D(32, (3, 3), activation='relu')(input_tensor)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.3)(x)  # Dropout between 0.3–0.5 work well

In [11]:
def build_vgg():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(132, 132, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [12]:
def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)

    # Match dimensions if needed
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)

    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_resnet():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D()(x)
    x = residual_block(x, 64)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [13]:
def build_mobilenet_style():
    model = models.Sequential([
        layers.SeparableConv2D(32, (3, 3), activation='relu', input_shape=(132, 132, 3)),
        layers.MaxPooling2D(),
        layers.SeparableConv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [14]:
def build_bn_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(132, 132, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [15]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout

x = Conv2D(32, (3, 3), activation='relu')(input_tensor)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.3)(x)  # Dropout between 0.3–0.5 tends to work well

In [16]:
# from sklearn.model_selection import train_test_split, GridSearchCV

# # Load arrays
# images = np.load('resized_images.npy')
# labels = np.load('labels.npy')

# # Train-test split
# x_train, x_test, y_train, y_test = train_test_split(
#     images, labels, test_size=0.2, random_state=42, stratify=labels
# )

# parameters = {'epoch': [10, 20, 30, 40], 'batch_size': [30, 40, 50, 60]}

# best_model = GridSearchCV(build_mobilenet_style(), param_grid = parameters, scoring= 'accuracy')
# best_model.fit(x_train, y_train)
# print(best_model.best_params_)

In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score
models_to_run = {
    'LeNet': build_lenet(),
    'VGG': build_vgg(),
    'ResNet': build_resnet(),
    'MobileNet-ish': build_mobilenet_style(),
    'BN-CNN': build_bn_cnn()
}


In [ ]:
from sklearn.model_selection import train_test_split

# Load arrays
images = np.load('resized_images.npy')
labels = np.load('labels.npy')

# Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

# List your models
models_to_run = {
    'LeNet': build_lenet(),
    'VGG': build_vgg(),
    'ResNet': build_resnet(),
    'MobileNet-ish': build_mobilenet_style(),
    'BN-CNN': build_bn_cnn()
}

# Train & evaluate each
for name, model in models_to_run.items():
    print(f"\n Training {name}...")
    model.fit(x_train, y_train, epochs=50, batch_size=32, validation_split=0.1)
    loss, acc = model.evaluate(x_test, y_test)
    print(f"{name} Test Accuracy: {acc:.4f}")

C:\Users\akank\anaconda3\envs\myenv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\akank\anaconda3\envs\myenv\Lib\site-packages\keras\src\layers\convolutional\base_separable_conv.py:104: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(



 Training LeNet...
Epoch 1/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 183ms/step - accuracy: 0.4546 - loss: 0.8907 - val_accuracy: 0.5125 - val_loss: 0.6803
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 168ms/step - accuracy: 0.5511 - loss: 0.6884 - val_accuracy: 0.5625 - val_loss: 0.6592
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 164ms/step - accuracy: 0.6630 - loss: 0.6167 - val_accuracy: 0.7000 - val_loss: 0.5850
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 167ms/step - accuracy: 0.7764 - loss: 0.4961 - val_accuracy: 0.7875 - val_loss: 0.4249
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 160ms/step - accuracy: 0.7439 - loss: 0.5184 - val_accuracy: 0.8125 - val_loss: 0.3950
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 157ms/step - accuracy: 0.7843 - loss: 0.4208 - val_accuracy: 0.8250 - val_loss: 0.4004
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 151ms/step - accuracy: 0.7830 - loss: 0.4045 - val_accuracy: 0.8500 - val_loss: 0.3566
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 165ms/step - accuracy: 0.8111 - loss: 0.393

In [16]:
from sklearn.decomposition import PCA
import numpy as np

def apply_columnwise_pca(image, n_components=132):
    compressed_channels = []

    for c in range(3):  # R, G, B
        channel = image[:, :, c]  # shape: (m, n)
        pca = PCA(n_components=n_components)
        transformed = pca.fit_transform(channel)  # shape: (m, n_components)
        compressed_channels.append(transformed)

    return np.stack(compressed_channels, axis=-1)  # shape: (m, n_components, 3)

In [17]:
from sklearn.decomposition import PCA
import numpy as np

def double_pca_channel(channel, n_components=132):
    # Step 1: Column-wise PCA
    pca_col = PCA(n_components=n_components)
    col_compressed = pca_col.fit_transform(channel)  # shape: (m, 132)

    # Step 2: Transpose
    transposed = col_compressed.T  # shape: (132, m)

    # Step 3: Row-wise PCA
    pca_row = PCA(n_components=n_components)
    row_compressed = pca_row.fit_transform(transposed)  # shape: (132, 132)

    # Step 4: Transpose back
    final_channel = row_compressed.T  # shape: (132, 132)

    return final_channel

In [18]:
def double_pca_image(image, n_components=132):
    compressed_channels = []

    for c in range(3):  # R, G, B
        channel = image[:, :, c]
        compressed = double_pca_channel(channel, n_components)
        compressed_channels.append(compressed)

    return np.stack(compressed_channels, axis=-1)  # shape: (132, 132, 3)

In [19]:
compressed_images = np.array([double_pca_image(img) for img in image_array])

In [20]:
import numpy as np

X_raw = np.load('resized_images.npy')  # shape: (num_images, 132, 132, 3)
y_raw = np.load('labels.npy')          # shape: (num_images,)

In [21]:
from sklearn.decomposition import PCA

def double_pca_channel(channel, n_components=132):
    pca_col = PCA(n_components=n_components)
    reduced_col = pca_col.fit_transform(channel)
    transposed = reduced_col.T
    pca_row = PCA(n_components=n_components)
    reduced_row = pca_row.fit_transform(transposed)
    return reduced_row.T

def compress_image_double_pca(image, n_components=132):
    R = double_pca_channel(image[:, :, 0], n_components)
    G = double_pca_channel(image[:, :, 1], n_components)
    B = double_pca_channel(image[:, :, 2], n_components)
    return np.stack([R, G, B], axis=-1)

In [22]:
X_pca = np.array([compress_image_double_pca(img) for img in X_raw])
y_pca = y_raw.copy()  # Labels stay the same

In [23]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np 

def build_lenet(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(6, kernel_size=5, activation='relu', input_shape=input_shape),
        tf.keras.layers.AveragePooling2D(pool_size=(2, 2)),
        tf.keras.layers.Conv2D(16, kernel_size=5, activation='relu'),
        tf.keras.layers.AveragePooling2D(pool_size=(2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(120, activation='relu'),
        tf.keras.layers.Dense(84, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_vgg(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_resnet(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    shortcut = x
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_mobilenet(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 3, strides=1, padding='same', activation='relu', input_shape=input_shape),
        tf.keras.layers.DepthwiseConv2D(3, padding='same', activation='relu'),
        tf.keras.layers.Conv2D(64, 1, activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_bncnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=input_shape),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model
    

def train_model(X, y, model_fn, label):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = model_fn(X_train.shape[1:], len(np.unique(y)))
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    print(f"{label} Accuracy: {acc:.4f}")
    return acc


model_fns = {
    'LeNet': build_lenet,
    'VGG': build_vgg,
    'ResNet': build_resnet,
    'MobileNet': build_mobilenet,
    'BN-CNN': build_bncnn,
}

results = {}

for name, fn in model_fns.items():
    print(f"\n Training {name} on RAW images...")
    acc_raw = train_model(X_raw, y_raw, fn, f"{name} (Raw)")

    print(f" Training {name} on PCA-compressed images...")
    acc_pca = train_model(X_pca, y_pca, fn, f"{name} (PCA)")




 Training LeNet on RAW images...


C:\Users\akank\anaconda3\envs\myenv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


LeNet (Raw) Accuracy: 0.8500
 Training LeNet on PCA-compressed images...
LeNet (PCA) Accuracy: 0.5900

 Training VGG on RAW images...



KeyboardInterrupt



In [24]:
from tensorflow.keras.preprocessing.image import array_to_img, img_to_array

def preprocess_gradient_images(images, target_size=(32, 32)):
    processed = []
    for img in images:
        # Convert to grayscale if needed
        if img.ndim == 3 and img.shape[-1] == 3:
            img = np.mean(img, axis=-1)

        # Computing gradient magnitude
        grad_x = np.diff(img, axis=1, prepend=img[:, :1])
        grad_y = np.diff(img, axis=0, prepend=img[:1, :])
        grad = np.sqrt(grad_x**2 + grad_y**2)

        # Normalize and resize
        grad = (grad / grad.max()) * 255
        grad = grad.astype(np.uint8)
        grad_img = array_to_img(grad[..., np.newaxis])
        grad_resized = img_to_array(grad_img.resize(target_size))

        processed.append(grad_resized)
    return np.array(processed).astype(np.float32) / 255.0

In [25]:
X_grad32 = preprocess_gradient_images(X_raw, target_size=(32, 32))
y_grad32 = y_raw.copy()

In [26]:
def train_model(X, y, model_fn, label):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = model_fn((32, 32, 1), len(np.unique(y)))
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)  # ← changed here
    acc = model.evaluate(X_test, y_test, verbose=0)[1]
    print(f"{label} Accuracy: {acc:.4f}")
    return acc

In [27]:
for name, fn in model_fns.items():
    print(f"\n Training {name} on GRADIENT-resized images with 50 epochs...")
    acc_grad32 = train_model(X_grad32, y_grad32, fn, f"{name} (Gradient 32x32, 50 epochs)")
    results.setdefault(name, {})['Gradient_32x32_50epochs'] = acc_grad32


 Training LeNet on GRADIENT-resized images with 50 epochs...
LeNet (Gradient 32x32, 50 epochs) Accuracy: 0.7150

 Training VGG on GRADIENT-resized images with 50 epochs...
VGG (Gradient 32x32, 50 epochs) Accuracy: 0.7600

 Training ResNet on GRADIENT-resized images with 50 epochs...
ResNet (Gradient 32x32, 50 epochs) Accuracy: 0.7600

 Training MobileNet on GRADIENT-resized images with 50 epochs...
MobileNet (Gradient 32x32, 50 epochs) Accuracy: 0.7050

 Training BN-CNN on GRADIENT-resized images with 50 epochs...
BN-CNN (Gradient 32x32, 50 epochs) Accuracy: 0.8000


# Resnet, Mobilenet and Bn Cnn fusion

In [75]:
from tensorflow.keras import layers, models


def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x



def build_resnet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = residual_block(x, 32)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D()(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)   # return features
    model = models.Model(inputs, x, name="resnet_features")
    return model



def build_mobilenet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.SeparableConv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.SeparableConv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.SeparableConv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.GlobalAveragePooling2D()(x)
    model = models.Model(inputs, x, name="mobilenet_features")
    return model



def build_bncnn_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    model = models.Model(inputs, x, name="bncnn_features")
    return model


In [76]:
def build_fusion_model():
    inputs = layers.Input(shape=(132, 132, 3))

    # Get features from each model
    resnet_feat = build_resnet_features()(inputs)
    mobilenet_feat = build_mobilenet_features()(inputs)
    bncnn_feat = build_bncnn_features()(inputs)

    # Concatenate features
    x = layers.Concatenate()([resnet_feat, bncnn_feat])  #  mobilenet_feat, 
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.15)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_model")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [77]:
fusion_model = build_fusion_model()
fusion_model.summary()  # See the architecture


Model: "fusion_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_62      │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_features     │ (None, 64)        │    171,200 │ input_layer_62[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_features      │ (None, 64)        │     19,776 │ input_layer_62[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 128)       │          0 │ resnet_features[… │
│ (Concatenate)       │                   │            │ bncnn_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_122 (Dense)   │ (None, 256)       │     33,024 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 256)       │          0 │ dense_122[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_123 (Dense)   │ (None, 1)         │        257 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 224,257 (876.00 KB)

 Trainable params: 223,105 (871.50 KB)

 Non-trainable params: 1,152 (4.50 KB)

In [78]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.load('resized_images.npy')   # shape: (1000, 132, 132, 3)
y = np.load('labels.npy')           # shape: (1000,)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train/Val/Test split (70% train, 15% val, 15% test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


X shape: (1000, 132, 132, 3)
y shape: (1000,)
Train: (700, 132, 132, 3) (700,)
Val: (150, 132, 132, 3) (150,)
Test: (150, 132, 132, 3) (150,)


In [79]:
fusion_model = build_fusion_model()
fusion_model.summary()

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)


Model: "fusion_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_66      │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_features     │ (None, 64)        │    171,200 │ input_layer_66[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_features      │ (None, 64)        │     19,776 │ input_layer_66[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 128)       │          0 │ resnet_features[… │
│ (Concatenate)       │                   │            │ bncnn_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_124 (Dense)   │ (None, 256)       │     33,024 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 256)       │          0 │ dense_124[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_125 (Dense)   │ (None, 1)         │        257 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 224,257 (876.00 KB)

 Trainable params: 223,105 (871.50 KB)

 Non-trainable params: 1,152 (4.50 KB)

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 86s 3s/step - accuracy: 0.7052 - loss: 0.4815 - val_accuracy: 0.5000 - val_loss: 1.0850
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 72s 3s/step - accuracy: 0.9145 - loss: 0.2190 - val_accuracy: 0.5067 - val_loss: 1.0077
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 72s 3s/step - accuracy: 0.8981 - loss: 0.2336 - val_accuracy: 0.5000 - val_loss: 0.9983
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 72s 3s/step - accuracy: 0.9432 - loss: 0.1556 - val_accuracy: 0.5800 - val_loss: 0.6284
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 72s 3s/step - accuracy: 0.9122 - loss: 0.2022 - val_accuracy: 0.7533 - val_loss: 0.6145
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 81s 3s/step - accuracy: 0.8999 - loss: 0.2514 - val_accuracy: 0.6733 - val_loss: 0.7352
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 83s 3s/step - accuracy: 0.8919 - loss: 0.2196 - val_accuracy: 0.5467 - val_loss: 1.6715
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 211s 3s/step - accuracy: 0.9465 - loss: 0.1399 - val_accuracy: 0.5400 - val_loss

In [80]:
test_loss, test_acc = fusion_model.evaluate(X_test, y_test)
print(f"✅ Test Accuracy: {test_acc:.4f}")


5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 548ms/step - accuracy: 0.6896 - loss: 1.0521
✅ Test Accuracy: 0.6600


# Same with Data Augmentation

In [85]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks


X = np.load("resized_images.npy")
y = np.load("labels.npy")

# Train/val/test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Data split:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# Data augmentation
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1),
])


def build_resnet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    return models.Model(inputs, x, name="resnet_feat")

def build_mobilenet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.SeparableConv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    return models.Model(inputs, x, name="mobilenet_feat")

def build_bncnn_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    return models.Model(inputs, x, name="bncnn_feat")

#Fusion model
def build_fusion_model():
    inputs = layers.Input(shape=(132, 132, 3))
    x = data_augmentation(inputs)  # apply augmentation here

    resnet_feat = build_resnet_features()(x)
    mobilenet_feat = build_mobilenet_features()(x)
    bncnn_feat = build_bncnn_features()(x)

    concat = layers.Concatenate()([resnet_feat, bncnn_feat]) # [resnet, vgg], [resnet, vgg, bncnn], [vgg, bncnn]
    x = layers.Dense(256, activation='relu')(concat)
    x = layers.Dropout(0.1)(x)
    # add more layers
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_model")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

fusion_model = build_fusion_model()
fusion_model.summary()

#Callbacks(building block for deep learning models in keras)

# early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)   #(Stops training early if the model is no longer improving, and keeps the best version of the model)
# checkpoint = callbacks.ModelCheckpoint("best_fusion_model.keras", save_best_only=True, monitor='val_accuracy')

# #Training
# history = fusion_model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=100,
#     batch_size=32,
#     callbacks=[early_stop, checkpoint]
# )
history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy with Data Augmentation: {test_acc:.4f}")


Data split:
Train: (700, 132, 132, 3) Val: (150, 132, 132, 3) Test: (150, 132, 132, 3)


Model: "fusion_model_deep"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_90      │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_20       │ (None, 132, 132,  │          0 │ input_layer_90[0… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_feat         │ (None, 64)        │  8,653,760 │ sequential_20[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_feat          │ (None, 64)        │  8,653,888 │ sequential_20[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_9       │ (None, 128)       │          0 │ resnet_feat[0][0… │
│ (Concatenate)       │                   │            │ bncnn_feat[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_152 (Dense)   │ (None, 512)       │     66,048 │ concatenate_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 512)       │          0 │ dense_152[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_153 (Dense)   │ (None, 256)       │    131,328 │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 256)       │          0 │ dense_153[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_154 (Dense)   │ (None, 128)       │     32,896 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 128)       │          0 │ dense_154[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_155 (Dense)   │ (None, 1)         │        129 │ dropout_16[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 17,538,049 (66.90 MB)

 Trainable params: 17,537,985 (66.90 MB)

 Non-trainable params: 64 (256.00 B)

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 19s 545ms/step - accuracy: 0.6355 - loss: 1.7219 - val_accuracy: 0.7667 - val_loss: 0.5907
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 11s 480ms/step - accuracy: 0.7356 - loss: 0.5672 - val_accuracy: 0.8200 - val_loss: 0.5764
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 457ms/step - accuracy: 0.8117 - loss: 0.3609 - val_accuracy: 0.7133 - val_loss: 0.5568
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 11s 479ms/step - accuracy: 0.8128 - loss: 0.3676 - val_accuracy: 0.7000 - val_loss: 0.5157
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 21s 476ms/step - accuracy: 0.8770 - loss: 0.3005 - val_accuracy: 0.7467 - val_loss: 0.4523
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 10s 472ms/step - accuracy: 0.8793 - loss: 0.3245 - val_accuracy: 0.7000 - val_loss: 0.5153
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 11s 484ms/step - accuracy: 0.9060 - loss: 0.2440 - val_accuracy: 0.7667 - val_loss: 0.4781
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 11s 479ms/step - accuracy: 0.8826 - loss: 0.3108 - val_accu

In [13]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt

#Load Data
X = np.load('resized_images.npy')
y = np.load('labels.npy')

# Train/val/test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Data split:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

#Data Augmentation
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
])

# Feature Extractors(resnet, mobilenet and bn cnn)
def build_resnet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="resnet_feat")

def build_mobilenet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.SeparableConv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.SeparableConv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="mobilenet_feat")

def build_bncnn_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="bncnn_feat")

#Fusion Model
def build_fusion_model():
    inputs = layers.Input(shape=(132, 132, 3))
    x = data_augmentation(inputs)

    resnet_feat = build_resnet_features()(x)
    mobilenet_feat = build_mobilenet_features()(x)
    bncnn_feat = build_bncnn_features()(x)

    concat = layers.Concatenate()([resnet_feat, mobilenet_feat, bncnn_feat])
    x = layers.Dense(256, activation='relu')(concat)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_model")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

#Train
fusion_model = build_fusion_model()
fusion_model.summary()

#early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
#checkpoint = callbacks.ModelCheckpoint("best_fusion_model.keras", save_best_only=True, monitor='val_accuracy')

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    #callbacks=[early_stop, checkpoint]
)

#Evaluate
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy with Fusion + Augmentation: {test_acc:.4f}")


Data split:
Train: (700, 132, 132, 3) Val: (150, 132, 132, 3) Test: (150, 132, 132, 3)


Model: "fusion_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 132, 132,  │          0 │ input_layer_1[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_feat         │ (None, 128)       │     36,960 │ sequential[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenet_feat      │ (None, 128)       │     10,875 │ sequential[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_feat          │ (None, 128)       │     28,096 │ sequential[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ resnet_feat[0][0… │
│ (Concatenate)       │                   │            │ mobilenet_feat[0… │
│                     │                   │            │ bncnn_feat[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │     98,560 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │     32,896 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │        129 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 207,516 (810.61 KB)

 Trainable params: 207,324 (809.86 KB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 52s 3s/step - accuracy: 0.6542 - loss: 0.6113 - val_accuracy: 0.5000 - val_loss: 0.6926
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 36s 3s/step - accuracy: 0.8052 - loss: 0.3799 - val_accuracy: 0.5000 - val_loss: 0.7722
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 37s 3s/step - accuracy: 0.8491 - loss: 0.3328 - val_accuracy: 0.5000 - val_loss: 0.7508
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 31s 3s/step - accuracy: 0.9064 - loss: 0.2484 - val_accuracy: 0.5133 - val_loss: 0.6808
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 40s 3s/step - accuracy: 0.9046 - loss: 0.2384 - val_accuracy: 0.5267 - val_loss: 0.6829
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 30s 3s/step - accuracy: 0.8861 - loss: 0.2662 - val_accuracy: 0.5333 - val_loss: 0.6232
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 31s 3s/step - accuracy: 0.9338 - loss: 0.1950 - val_accuracy: 0.7467 - val_loss: 0.5901
Epoch 8/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 29s 3s/step - accuracy: 0.9466 - loss: 0.1480 - val_accuracy: 0.7733 - v

# Resnet, Vgg fusion

In [87]:
from tensorflow.keras import layers, models

# ResNet Feature Extractor
def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)

    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_resnet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D()(x)
    x = residual_block(x, 64)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)  # feature embedding
    return models.Model(inputs, x, name="resnet_features")

# VGG Feature Extractor
def build_vgg_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)  # feature embedding
    return models.Model(inputs, x, name="vgg_features")

# Fusion Model
def build_fusion_model():
    inputs = layers.Input(shape=(132, 132, 3))

    # Extract features
    resnet_feat = build_resnet_features()(inputs)
    vgg_feat = build_vgg_features()(inputs)

    # Concatenate feature embeddings
    x = layers.Concatenate()([resnet_feat, vgg_feat])

    # Fully connected layers for classification
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_resnet_vgg")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build and summarize
fusion_model = build_fusion_model()
fusion_model.summary()


Model: "fusion_resnet_vgg"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_95      │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_features     │ (None, 128)       │  8,465,664 │ input_layer_95[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg_features        │ (None, 128)       │  7,438,496 │ input_layer_95[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_10      │ (None, 256)       │          0 │ resnet_features[… │
│ (Concatenate)       │                   │            │ vgg_features[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_158 (Dense)   │ (None, 256)       │     65,792 │ concatenate_10[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 256)       │          0 │ dense_158[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_159 (Dense)   │ (None, 128)       │     32,896 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 128)       │          0 │ dense_159[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_160 (Dense)   │ (None, 1)         │        129 │ dropout_18[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,002,977 (61.05 MB)

 Trainable params: 16,002,977 (61.05 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

# Load Data
X = np.load("resized_images.npy")  # shape: (1000, 132, 132, 3)
y = np.load("labels.npy")          # shape: (1000,)

# Train/val/test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# Data Augmentation
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

# Resnet and vgg
def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_resnet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D()(x)
    x = residual_block(x, 64)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="resnet_features")

def build_vgg_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="vgg_features")

# Fusion Model
def build_fusion_model():
    inputs = layers.Input(shape=(132, 132, 3))
    x = data_augmentation(inputs)  # apply augmentation

    resnet_feat = build_resnet_features()(x)
    vgg_feat = build_vgg_features()(x)

    concat = layers.Concatenate()([resnet_feat, vgg_feat])
    x = layers.Dense(256, activation='relu')(concat)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_resnet_vgg")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Train 
fusion_model = build_fusion_model()
fusion_model.summary()

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,      
    batch_size=32
)

# Evaluate
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy with Fusion + Data Augmentation: {test_acc:.4f}")


Train: (700, 132, 132, 3) Val: (150, 132, 132, 3) Test: (150, 132, 132, 3)


Model: "fusion_resnet_vgg"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 132, 132,  │          0 │ input_layer_6[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_features     │ (None, 128)       │  8,465,664 │ sequential_1[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg_features        │ (None, 128)       │  7,438,496 │ sequential_1[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 256)       │          0 │ resnet_features[… │
│ (Concatenate)       │                   │            │ vgg_features[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 256)       │     65,792 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 128)       │     32,896 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │        129 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,002,977 (61.05 MB)

 Trainable params: 16,002,977 (61.05 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.4941 - loss: 0.8399 - val_accuracy: 0.6733 - val_loss: 0.5823
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step - accuracy: 0.6685 - loss: 0.6053 - val_accuracy: 0.7267 - val_loss: 0.5085
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.7020 - loss: 0.5341 - val_accuracy: 0.7933 - val_loss: 0.3685
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.7761 - loss: 0.4375 - val_accuracy: 0.8000 - val_loss: 0.3717
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.7975 - loss: 0.4219 - val_accuracy: 0.7267 - val_loss: 0.9512
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.7595 - loss: 0.5161 - val_accuracy: 0.8000 - val_loss: 0.3322
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.7924 - loss: 0.4232 - val_accuracy: 0.8400 - val_loss: 0.3458
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.7890 - loss: 0.4083 - val_accuracy: 0.8133 - v

# Resnet, Vgg and Bn Cnn fusion

In [21]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

# Load Data
X = np.load("resized_images.npy")  # shape: (1000, 132, 132, 3)
y = np.load("labels.npy")          # shape: (1000,)

# Train/val/test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# Data Augmentation
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_resnet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D()(x)
    x = residual_block(x, 64)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="resnet_features")

def build_vgg_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="vgg_features")

def build_bncnn_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="bncnn_features")

# Fusion Model
def build_fusion_model():
    inputs = layers.Input(shape=(132, 132, 3))
    x = data_augmentation(inputs)  # apply augmentation

    resnet_feat = build_resnet_features()(x)
    vgg_feat = build_vgg_features()(x)
    bncnn_feat = build_bncnn_features()(x)

    concat = layers.Concatenate()([resnet_feat, vgg_feat, bncnn_feat])
    x = layers.Dense(256, activation='relu')(concat)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_resnet_vgg_bncnn")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Train
fusion_model = build_fusion_model()
fusion_model.summary()

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,      # increasing epochs for better accuracy
    batch_size=32
)

# Evaluate
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy with ResNet + VGG + BN-CNN Fusion: {test_acc:.4f}")


Train: (700, 132, 132, 3) Val: (150, 132, 132, 3) Test: (150, 132, 132, 3)


Model: "fusion_resnet_vgg_bncnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_11      │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_5        │ (None, 132, 132,  │          0 │ input_layer_11[0… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_features     │ (None, 128)       │  8,465,664 │ sequential_5[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg_features        │ (None, 128)       │  7,438,496 │ sequential_5[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_features      │ (None, 128)       │  7,892,416 │ sequential_5[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 384)       │          0 │ resnet_features[… │
│ (Concatenate)       │                   │            │ vgg_features[0][… │
│                     │                   │            │ bncnn_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 256)       │     98,560 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ dense_19[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 128)       │     32,896 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 1)         │        129 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,928,161 (91.28 MB)

 Trainable params: 23,927,969 (91.28 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 0.7023 - loss: 2.1003 - val_accuracy: 0.5000 - val_loss: 1.6997
Epoch 2/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.7390 - loss: 1.2157 - val_accuracy: 0.5000 - val_loss: 0.9364
Epoch 3/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.7669 - loss: 1.0443 - val_accuracy: 0.5000 - val_loss: 1.5667
Epoch 4/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.8059 - loss: 0.6767 - val_accuracy: 0.5000 - val_loss: 1.0422
Epoch 5/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.8299 - loss: 0.5598 - val_accuracy: 0.6933 - val_loss: 0.5800
Epoch 6/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 62s 3s/step - accuracy: 0.8332 - loss: 0.5330 - val_accuracy: 0.6867 - val_loss: 0.6680
Epoch 7/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.8666 - loss: 0.3372 - val_accuracy: 0.8133 - val_loss: 0.4492
Epoch 8/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step - accuracy: 0.8726 - loss: 0.3537 - val_accuracy: 0.6067 - val_loss:

# ResNet + VGG + BN-CNN Fusion + Strong Augmentation

In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks

# Load Data
X = np.load("resized_images.npy")   # shape: (1000, 132, 132, 3)
y = np.load("labels.npy")           # shape: (1000,)

# Train/Val/Test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# Data Augmentation
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.3),
    layers.RandomTranslation(0.2, 0.2),
    layers.RandomContrast(0.15),
])

# Residual Block
def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

# resnet, vgg and bn cnn
def build_resnet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D()(x)
    x = residual_block(x, 64)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="resnet_feat")

def build_vgg_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="vgg_feat")

def build_bncnn_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="bncnn_feat")

# Fusion Model
def build_fusion_model():
    inputs = layers.Input(shape=(132,132,3))
    x = data_augmentation(inputs)  # apply augmentation

    resnet_feat = build_resnet_features()(x)
    vgg_feat = build_vgg_features()(x)
    bncnn_feat = build_bncnn_features()(x)

    # Concatenate all features
    concat = layers.Concatenate()([resnet_feat, vgg_feat, bncnn_feat])
    x = layers.Dense(512, activation='relu')(concat)
    x = layers.Dropout(0.15)(x)  # dropout=0.1/0.2/none
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.15)(x) # dropout=0.1/0.15/none
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.15)(x) # dropout=0.1/0.15/none
    #x = layers.Dense(64, activation='relu')(x)
    #x = layers.Dropout(0.1)(x) # dropout=0.1/0.15/none
    #x = layers.Dense(32, activation='relu')(x)
    #x = layers.Dropout(0.1)(x) # dropout=0.1/0.15/none
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_model")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

#Train
fusion_model = build_fusion_model()
fusion_model.summary()

#early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
#checkpoint = callbacks.ModelCheckpoint("best_fusion_model.keras", save_best_only=True, monitor='val_accuracy')

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=70, # epoch=70,100
    batch_size=64, # batch_size=64
    #callbacks=[early_stop, checkpoint]
)

#Evaluate
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy with ResNet + VGG + BN-CNN Fusion + Strong Augmentation: {test_acc:.4f}")


Train: (700, 132, 132, 3) Val: (150, 132, 132, 3) Test: (150, 132, 132, 3)


Model: "fusion_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 132, 132,  │          0 │ input_layer_6[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_feat         │ (None, 128)       │     85,248 │ sequential_1[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg_feat            │ (None, 128)       │     73,888 │ sequential_1[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_feat          │ (None, 128)       │     28,096 │ sequential_1[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 384)       │          0 │ resnet_feat[0][0… │
│ (Concatenate)       │                   │            │ vgg_feat[0][0],   │
│                     │                   │            │ bncnn_feat[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 512)       │    197,120 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 512)       │          0 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 256)       │    131,328 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 256)       │          0 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 128)       │     32,896 │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 128)       │          0 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 1)         │        129 │ dropout_9[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 548,705 (2.09 MB)

 Trainable params: 548,513 (2.09 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.7160 - loss: 0.5372 - val_accuracy: 0.5000 - val_loss: 0.7377
Epoch 2/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 36s 3s/step - accuracy: 0.8473 - loss: 0.3574 - val_accuracy: 0.5000 - val_loss: 0.7117
Epoch 3/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 36s 3s/step - accuracy: 0.8873 - loss: 0.3037 - val_accuracy: 0.5133 - val_loss: 0.6818
Epoch 4/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 36s 3s/step - accuracy: 0.8966 - loss: 0.2643 - val_accuracy: 0.5200 - val_loss: 0.6632
Epoch 5/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.9224 - loss: 0.1967 - val_accuracy: 0.5133 - val_loss: 0.6936
Epoch 6/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 38s 3s/step - accuracy: 0.9185 - loss: 0.2055 - val_accuracy: 0.6200 - val_loss: 0.6057
Epoch 7/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 46s 4s/step - accuracy: 0.9052 - loss: 0.2176 - val_accuracy: 0.7733 - val_loss: 0.5811
Epoch 8/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 36s 3s/step - accuracy: 0.9266 - loss: 0.1856 - val_accuracy: 0.8267 - val_loss:

# ResNet + VGG + BN-CNN (Average Fusion)

In [21]:
#Fusion Model with Average (Deeper)
def build_fusion_model_average_deep():
    inputs = layers.Input(shape=(132,132,3))
    x = data_augmentation(inputs)  # apply augmentation

    resnet_feat = build_resnet_features()(x)
    vgg_feat = build_vgg_features()(x)
    bncnn_feat = build_bncnn_features()(x)

    # Average fusion instead of concatenation
    avg = layers.Average()([resnet_feat, vgg_feat, bncnn_feat])

    # Added more layers
    x = layers.Dense(1024, activation='relu')(avg)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.1)(x)

    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.05)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_model_avg_deep")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [22]:
#Train
fusion_model_avg_deep = build_fusion_model_average_deep()
fusion_model_avg_deep.summary()

# Callbacks (early stopping + best model saving)
#early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
#checkpoint = callbacks.ModelCheckpoint("best_fusion_avg_deep_model.keras", save_best_only=True, monitor='val_accuracy')

# Train the model
history = fusion_model_avg_deep.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,          # can increase to 100 if needed
    batch_size=32,
    #callbacks=[early_stop, checkpoint]
)

# Evaluate
test_loss, test_acc = fusion_model_avg_deep.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy with ResNet + VGG + BN-CNN (Average Fusion + Deep Head): {test_acc:.4f}")


Model: "fusion_model_avg_deep"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_29      │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_8        │ (None, 132, 132,  │          0 │ input_layer_29[0… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_feat         │ (None, 128)       │     85,248 │ sequential_8[1][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg_feat            │ (None, 128)       │     73,888 │ sequential_8[1][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_feat          │ (None, 128)       │     28,096 │ sequential_8[1][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average (Average)   │ (None, 128)       │          0 │ resnet_feat[0][0… │
│                     │                   │            │ vgg_feat[0][0],   │
│                     │                   │            │ bncnn_feat[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_43 (Dense)    │ (None, 1024)      │    132,096 │ average[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1024)      │      4,096 │ dense_43[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 1024)      │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_44 (Dense)    │ (None, 512)       │    524,800 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_44[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 512)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_45 (Dense)    │ (None, 256)       │    131,328 │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_45[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 256)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_46 (Dense)    │ (None, 128)       │     32,896 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_46[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 128)       │          0 │ batch_normalizat

 Total params: 1,026,657 (3.92 MB)

 Trainable params: 1,022,497 (3.90 MB)

 Non-trainable params: 4,160 (16.25 KB)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step - accuracy: 0.6266 - loss: 0.6864 - val_accuracy: 0.5000 - val_loss: 0.7003
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.8897 - loss: 0.2928 - val_accuracy: 0.5000 - val_loss: 0.7546
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 35s 2s/step - accuracy: 0.8800 - loss: 0.3493 - val_accuracy: 0.5000 - val_loss: 0.7113
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step - accuracy: 0.8983 - loss: 0.2778 - val_accuracy: 0.5000 - val_loss: 0.7142
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.8612 - loss: 0.3140 - val_accuracy: 0.5000 - val_loss: 0.8182
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.8914 - loss: 0.2850 - val_accuracy: 0.5000 - val_loss: 0.7878
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.8769 - loss: 0.2883 - val_accuracy: 0.5000 - val_loss: 0.7401
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.9034 - loss: 0.2327 - val_accuracy: 0.5000 - val_loss:

# Maximum Fusion(resnet+vgg+bn cnn)

In [22]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks

# ------------------------------
# Load Data
# ------------------------------
X = np.load("resized_images.npy")   # shape: (1000, 132, 132, 3)
y = np.load("labels.npy")           # shape: (1000,)

# Train/Val/Test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# ------------------------------
# Data Augmentation
# ------------------------------
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.3),
    layers.RandomTranslation(0.2, 0.2),
    layers.RandomContrast(0.15),
])

# ------------------------------
# Custom Feature Extractors
# ------------------------------
def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_resnet_features():
    inputs = layers.Input(shape=(132, 132, 3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D()(x)
    x = residual_block(x, 64)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="resnet_feat")

def build_vgg_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="vgg_feat")

def build_bncnn_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="bncnn_feat")

# ------------------------------
# Fusion Model with Maximum Fusion
# ------------------------------
def build_fusion_model_max_deep():
    inputs = layers.Input(shape=(132,132,3))
    x = data_augmentation(inputs)  # apply augmentation

    resnet_feat = build_resnet_features()(x)
    vgg_feat = build_vgg_features()(x)
    bncnn_feat = build_bncnn_features()(x)

    # Maximum fusion instead of Concatenate/Average
    fused = layers.Maximum()([resnet_feat, vgg_feat, bncnn_feat])

    # Fully connected layers
    x = layers.Dense(1024, activation='relu')(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.1)(x)

    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.05)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_model_max_deep")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ------------------------------
# Train the Fusion Model
# ------------------------------
fusion_model = build_fusion_model_max_deep()
fusion_model.summary()

#early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
#checkpoint = callbacks.ModelCheckpoint("best_fusion_model_max.keras", save_best_only=True, monitor='val_accuracy')

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    #callbacks=[early_stop, checkpoint],
    verbose=1
)

# ------------------------------
# Evaluate
# ------------------------------
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy with ResNet + VGG + BN-CNN (Max Fusion + Deep FC): {test_acc:.4f}")


Train: (700, 132, 132, 3) Val: (150, 132, 132, 3) Test: (150, 132, 132, 3)


Model: "fusion_model_max_deep"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_20      │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_3        │ (None, 132, 132,  │          0 │ input_layer_20[0… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_feat         │ (None, 128)       │     85,248 │ sequential_3[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg_feat            │ (None, 128)       │     73,888 │ sequential_3[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_feat          │ (None, 128)       │     28,096 │ sequential_3[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ maximum (Maximum)   │ (None, 128)       │          0 │ resnet_feat[0][0… │
│                     │                   │            │ vgg_feat[0][0],   │
│                     │                   │            │ bncnn_feat[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 1024)      │    132,096 │ maximum[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1024)      │      4,096 │ dense_21[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 1024)      │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 512)       │    524,800 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_22[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 512)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 256)       │    131,328 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_23[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 256)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 128)       │     32,896 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_24[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 128)       │          0 │ batch_normalizat

 Total params: 1,026,657 (3.92 MB)

 Trainable params: 1,022,497 (3.90 MB)

 Non-trainable params: 4,160 (16.25 KB)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 69s 4s/step - accuracy: 0.6137 - loss: 0.6569 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 53s 5s/step - accuracy: 0.8377 - loss: 0.3410 - val_accuracy: 0.5000 - val_loss: 0.6959
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 44s 4s/step - accuracy: 0.9033 - loss: 0.2320 - val_accuracy: 0.5000 - val_loss: 0.7098
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step - accuracy: 0.9123 - loss: 0.2033 - val_accuracy: 0.5000 - val_loss: 0.7310
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 56s 5s/step - accuracy: 0.9168 - loss: 0.2432 - val_accuracy: 0.5000 - val_loss: 0.7929
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 34s 3s/step - accuracy: 0.9276 - loss: 0.2075 - val_accuracy: 0.5000 - val_loss: 0.7900
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 0.9230 - loss: 0.1811 - val_accuracy: 0.5067 - val_loss: 0.7375
Epoch 8/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 43s 4s/step - accuracy: 0.9412 - loss: 0.1888 - val_accuracy: 0.5000 - v

# VGG, BN CNN

In [25]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks

#load data
X = np.load("resized_images.npy")   # shape: (1000, 132, 132, 3)
y = np.load("labels.npy")           # shape: (1000,)

# Train/Val/Test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

#Data Augmentation
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.3),
    layers.RandomTranslation(0.2, 0.2),
    layers.RandomContrast(0.15),
])

#vgg and bn cnn
def build_vgg_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="vgg_feat")

def build_bncnn_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    return models.Model(inputs, x, name="bncnn_feat")

#Fusion Model
def build_vgg_bncnn_fusion():
    inputs = layers.Input(shape=(132,132,3))
    x = data_augmentation(inputs)

    vgg_feat = build_vgg_features()(x)
    bncnn_feat = build_bncnn_features()(x)

    # more layers added
    concat = layers.Concatenate()([vgg_feat, bncnn_feat])

    x = layers.Dense(1024, activation='relu')(concat)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="vgg_bncnn_fusion_deep")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


#Train
fusion_model = build_vgg_bncnn_fusion()
fusion_model.summary()

#early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
#checkpoint = callbacks.ModelCheckpoint("best_vgg_bncnn_fusion.keras", save_best_only=True, monitor='val_accuracy')

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=70,
    batch_size=64,
    #callbacks=[early_stop, checkpoint]
)

#Evaluate
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Test Accuracy with VGG + BN-CNN Fusion: {test_acc:.4f}")


Train: (700, 132, 132, 3) Val: (150, 132, 132, 3) Test: (150, 132, 132, 3)


Model: "vgg_bncnn_fusion_deep"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_33      │ (None, 132, 132,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_6        │ (None, 132, 132,  │          0 │ input_layer_33[0… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg_feat            │ (None, 128)       │     73,888 │ sequential_6[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bncnn_feat          │ (None, 128)       │     28,096 │ sequential_6[0][… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 256)       │          0 │ vgg_feat[0][0],   │
│ (Concatenate)       │                   │            │ bncnn_feat[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_44 (Dense)    │ (None, 1024)      │    263,168 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1024)      │      4,096 │ dense_44[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_24          │ (None, 1024)      │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_45 (Dense)    │ (None, 512)       │    524,800 │ dropout_24[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_45[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_25          │ (None, 512)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_46 (Dense)    │ (None, 256)       │    131,328 │ dropout_25[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_46[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_26          │ (None, 256)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_47 (Dense)    │ (None, 128)       │     32,896 │ dropout_26[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_27          │ (None, 128)       │          0 │ dense_47[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_48 (Dense)    │ (None, 1)         │        129 │ dropout_27[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,061,473 (4.05 MB)

 Trainable params: 1,057,697 (4.03 MB)

 Non-trainable params: 3,776 (14.75 KB)

Epoch 1/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.7387 - loss: 0.5277 - val_accuracy: 0.7667 - val_loss: 0.6876
Epoch 2/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step - accuracy: 0.8941 - loss: 0.2412 - val_accuracy: 0.5000 - val_loss: 0.6883
Epoch 3/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.9177 - loss: 0.2030 - val_accuracy: 0.5000 - val_loss: 0.6981
Epoch 4/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.9240 - loss: 0.2112 - val_accuracy: 0.5000 - val_loss: 0.6907
Epoch 5/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.9118 - loss: 0.2428 - val_accuracy: 0.6133 - val_loss: 0.6608
Epoch 6/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 21s 2s/step - accuracy: 0.9013 - loss: 0.2766 - val_accuracy: 0.6133 - val_loss: 0.6642
Epoch 7/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.8915 - loss: 0.2161 - val_accuracy: 0.7267 - val_loss: 0.6552
Epoch 8/70
11/11 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.9383 - loss: 0.1583 - val_accuracy: 0.5333 - val_loss:

In [17]:
from tensorflow.keras import layers, models, regularizers, optimizers

# VGG
def build_vgg_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    return models.Model(inputs, x, name="vgg_feat")

# BN-CNN
def build_bncnn_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    return models.Model(inputs, x, name="bncnn_feat")

#Fusion
def build_vgg_bncnn_fusion():
    inputs = layers.Input(shape=(132,132,3))

    # Extract features
    vgg_feat = build_vgg_features()(inputs)
    bncnn_feat = build_bncnn_features()(inputs)

    # Concatenate
    concat = layers.Concatenate()([vgg_feat, bncnn_feat])

    # Fully connected layers
    x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(concat)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="vgg_bncnn_fusion")

    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [19]:
from sklearn.model_selection import train_test_split
import numpy as np

# loading dataset
X = np.load("resized_images.npy")   # shape: (num_samples, 132, 132, 3)
y = np.load("labels.npy")           # shape: (num_samples, )

# Split into train, validation, test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


In [20]:
fusion_model = build_vgg_bncnn_fusion()

#from tensorflow.keras import callbacks
#early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
#reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    #callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Test accuracy
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=1)
print(f"✅ Test Accuracy: {test_acc:.4f}")


Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.5680 - loss: 0.8747 - val_accuracy: 0.5000 - val_loss: 0.7778
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 22s 994ms/step - accuracy: 0.7817 - loss: 0.5448 - val_accuracy: 0.5000 - val_loss: 0.7804
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.8297 - loss: 0.4861 - val_accuracy: 0.5000 - val_loss: 0.7901
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step - accuracy: 0.8186 - loss: 0.4517 - val_accuracy: 0.5000 - val_loss: 0.8140
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.8229 - loss: 0.4622 - val_accuracy: 0.5000 - val_loss: 0.8488
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step - accuracy: 0.8664 - loss: 0.4275 - val_accuracy: 0.5000 - val_loss: 0.9200
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 39s 979ms/step - accuracy: 0.8575 - loss: 0.4238 - val_accuracy: 0.5000 - val_loss: 0.9506
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 22s 990ms/step - accuracy: 0.8558 - loss: 0.3948 - val_accuracy: 0

In [15]:
from tensorflow.keras import layers, models, regularizers, optimizers

# VGG
def build_vgg_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    return models.Model(inputs, x, name="vgg_feat")

# BN-CNN
def build_bncnn_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    return models.Model(inputs, x, name="bncnn_feat")

#Fusion
def build_vgg_bncnn_fusion():
    inputs = layers.Input(shape=(132,132,3))

    # Extract features
    vgg_feat = build_vgg_features()(inputs)
    bncnn_feat = build_bncnn_features()(inputs)

    # Concatenate
    concat = layers.Concatenate()([vgg_feat, bncnn_feat])

    # Fully connected layers
    x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(concat)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="vgg_bncnn_fusion")

    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [16]:
from sklearn.model_selection import train_test_split
import numpy as np

# loading dataset
X = np.load("resized_images.npy")   # shape: (num_samples, 132, 132, 3)
y = np.load("labels.npy")           # shape: (num_samples, )

# Split into train, validation, test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


In [18]:
fusion_model = build_vgg_bncnn_fusion()

#from tensorflow.keras import callbacks
#early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
#reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    #callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Test accuracy
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=1)
print(f"✅ Test Accuracy: {test_acc:.4f}")


Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 30s 2s/step - accuracy: 0.5916 - loss: 0.8961 - val_accuracy: 0.5000 - val_loss: 0.7778
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.7407 - loss: 0.5940 - val_accuracy: 0.5000 - val_loss: 0.7783
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 21s 2s/step - accuracy: 0.7842 - loss: 0.5337 - val_accuracy: 0.5000 - val_loss: 0.7805
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.8121 - loss: 0.4489 - val_accuracy: 0.5000 - val_loss: 0.7830
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - accuracy: 0.8265 - loss: 0.4421 - val_accuracy: 0.5000 - val_loss: 0.7861
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.8709 - loss: 0.3950 - val_accuracy: 0.5000 - val_loss: 0.7918
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.8750 - loss: 0.3636 - val_accuracy: 0.5000 - val_loss: 0.7979
Epoch 8/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.9102 - loss: 0.3254 - val_accuracy: 0.5000 - v

In [19]:
from tensorflow.keras import layers, models, regularizers, optimizers

# VGG
def build_vgg_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    return models.Model(inputs, x, name="vgg_feat")

# BN-CNN
def build_bncnn_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    return models.Model(inputs, x, name="bncnn_feat")

#Fusion
def build_vgg_bncnn_fusion():
    inputs = layers.Input(shape=(132,132,3))

    # Extract features
    vgg_feat = build_vgg_features()(inputs)
    bncnn_feat = build_bncnn_features()(inputs)

    # Concatenate
    concat = layers.Concatenate()([vgg_feat, bncnn_feat])

    # Fully connected layers
    x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(concat)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="vgg_bncnn_fusion")

    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [20]:
from sklearn.model_selection import train_test_split
import numpy as np

# loading dataset
X = np.load("resized_images.npy")   # shape: (num_samples, 132, 132, 3)
y = np.load("labels.npy")           # shape: (num_samples, )

# Split into train, validation, test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


In [ ]:
fusion_model = build_vgg_bncnn_fusion()

#from tensorflow.keras import callbacks
#early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
#reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    #callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Test accuracy
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=1)
print(f"✅ Test Accuracy: {test_acc:.4f}")


In [ ]:
from tensorflow.keras import layers, models, regularizers, optimizers
import tensorflow as tf

# VGG Features
def build_vgg_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), activation=None)(inputs)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Conv2D(32, (3,3), activation=None)(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3,3), activation=None)(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Conv2D(64, (3,3), activation=None)(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation=None)(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    return models.Model(inputs, x, name="vgg_feat")

# BN-CNN Features
def build_bncnn_features():
    inputs = layers.Input(shape=(132,132,3))
    x = layers.Conv2D(32, (3,3), padding='same', activation=None)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3,3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    return models.Model(inputs, x, name="bncnn_feat")

# Fusion Model
def build_vgg_bncnn_fusion():
    inputs = layers.Input(shape=(132,132,3))

    # Extract features
    vgg_feat = build_vgg_features()(inputs)
    bncnn_feat = build_bncnn_features()(inputs)

    # Concatenate features
    concat = layers.Concatenate()([vgg_feat, bncnn_feat])

    # Fully connected layers
    x = layers.Dense(1024, kernel_regularizer=regularizers.l2(1e-4))(concat)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(512, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="vgg_bncnn_fusion")

    # Cosine decay learning rate
    lr_schedule = optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=1e-3,
        first_decay_steps=50
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr_schedule),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

    return model


In [21]:
from tensorflow.keras import callbacks

# Build the model
fusion_model = build_vgg_bncnn_fusion()

# Callbacks
#early_stop = callbacks.EarlyStopping(monitor='val_accuracy',patience=15,  # wait for 15 epochs without improvementrestore_best_weights=True)

#checkpoint = callbacks.ModelCheckpoint("best_vgg_bncnn_fusion.keras",monitor='val_accuracy',save_best_only=True,verbose=1)

#reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7,min_lr=1e-6,verbose=1)

# Train
history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,         # give enough epochs for convergence
    batch_size=32,
    #callbacks=[early_stop, checkpoint, reduce_lr],
    verbose=1
)

# Evaluate on test data
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=1)
print(f"✅ Final Test Accuracy: {test_acc:.4f}")


Epoch 1/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.5706 - loss: 0.8984 - val_accuracy: 0.5000 - val_loss: 0.7769
Epoch 2/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.8064 - loss: 0.5433 - val_accuracy: 0.5000 - val_loss: 0.7770
Epoch 3/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.7980 - loss: 0.5024 - val_accuracy: 0.5000 - val_loss: 0.7773
Epoch 4/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 0.8207 - loss: 0.4333 - val_accuracy: 0.5000 - val_loss: 0.7768
Epoch 5/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.8272 - loss: 0.4506 - val_accuracy: 0.5000 - val_loss: 0.7925
Epoch 6/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.8720 - loss: 0.4053 - val_accuracy: 0.5000 - val_loss: 0.8150
Epoch 7/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.8939 - loss: 0.3642 - val_accuracy: 0.5000 - val_loss: 0.8146
Epoch 8/200
22/22 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.9069 - loss: 0.3263 - val_accuracy: 0.5000 - v

In [15]:
from sklearn.model_selection import train_test_split
import numpy as np

# Example: load your dataset (replace with your actual arrays)
X = np.load("resized_images.npy")   # shape: (num_samples, 132, 132, 3)
y = np.load("labels.npy")           # shape: (num_samples, )

# Split into train, validation, test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


In [16]:
import numpy as np
from tensorflow.keras import layers, models, regularizers, callbacks

# ✅ Data Augmentation Layer
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
], name="data_augmentation")


# ✅ VGG-like feature extractor
def build_vgg_features():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(256, activation='relu')
    ], name="vgg_features")
    return model


# ✅ BN-CNN feature extractor
def build_bncnn_features():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
    ], name="bncnn_features")
    return model


# ✅ Fusion Model
def build_vgg_bncnn_fusion():
    inputs = layers.Input(shape=(132, 132, 3))
    x = data_augmentation(inputs)  # Apply augmentation here

    vgg_feat = build_vgg_features()(x)
    bncnn_feat = build_bncnn_features()(x)

    concat = layers.Concatenate()([vgg_feat, bncnn_feat])

    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(concat)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name="fusion_vgg_bncnn")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


# ✅ Build and Train
fusion_model = build_vgg_bncnn_fusion()

# Optional callbacks
#early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
#reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

history = fusion_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    #callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ✅ Evaluate
test_loss, test_acc = fusion_model.evaluate(X_test, y_test, verbose=1)
print(f"✅ Test Accuracy with Fusion + Augmentation: {test_acc:.4f}")


Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 0.6790 - loss: 0.6719 - val_accuracy: 0.5000 - val_loss: 1.4087
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.8150 - loss: 0.4376 - val_accuracy: 0.5000 - val_loss: 4.0671
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.8793 - loss: 0.3533 - val_accuracy: 0.6000 - val_loss: 0.7455
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.8824 - loss: 0.3107 - val_accuracy: 0.5000 - val_loss: 0.8257
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.9137 - loss: 0.2827 - val_accuracy: 0.5067 - val_loss: 0.9322
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.9296 - loss: 0.2197 - val_accuracy: 0.5000 - val_loss: 1.3000
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step - accuracy: 0.9071 - loss: 0.2723 - val_accuracy: 0.5000 - val_loss: 0.9542
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 80s 2s/step - accuracy: 0.9182 - loss: 0.2289 - val_accuracy: 0.5000 - v

In [25]:
model_fns = {
    'LeNet': build_lenet,
    'VGG': build_vgg,
    'ResNet': build_resnet,
    'MobileNet': build_mobilenet,
    'BN-CNN': build_bncnn,}